# L3: Tools for a Customer Outreach Campaign

In this lesson, you will learn more about Tools. You'll focus on three key elements of Tools:
- Versatility
- Fault Tolerance
- Caching

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM
- [Serper](https://serper.dev)

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from utils import get_openai_api_key
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## Creating Agents

In [4]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

In [5]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

## Creating Tools

### crewAI Tools

In [6]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [7]:
directory_read_tool = DirectoryReadTool(directory='../../data/instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

### Custom Tool
- Create a custom tool using crewAi's [BaseTool](https://docs.crewai.com/core-concepts/Tools/#subclassing-basetool) class

In [8]:
from crewai.tools import BaseTool

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [9]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [10]:
sentiment_analysis_tool = SentimentAnalysisTool()

## Creating Tasks

- The Lead Profiling Task is using crewAI Tools.

In [11]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

- The Personalized Outreach Task is using your custom Tool `SentimentAnalysisTool`, as well as crewAI's `SerperDevTool` (search_tool).

In [12]:
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

## Creating the Crew

In [13]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [14]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bc358cb2-c8a6-4984-b76b-42d76398a0c7                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭────────────────────────────────────────────── 🧠 Retrieved Memory ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Historical Data:                                                                                               │
│  - Ensure detailed research on all key decision-makers, not just the founder, to provide a more holistic view.  │
│  - Include statistical data or case studies to support claims about market success and educational              │
│  effectiveness.                                                                                                 │
│  - Elaborate on the identified needs with concrete examples of how the solutions can directly address them.     │
│  - Provide a summary section to highlight key points for quick reference.                                       │
│  - Use visual aids or infographics to enhance the report's presentation and re...                               │
│                                                                                                                 │
╰─────────────────────────────────────────── Retrieval Time: 3295.03ms ───────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sales Representative                                                                                    │
│                                                                                                                 │
│  Task: Conduct an in-depth analysis of DeepLearningAI, a company in the Online Learning Platform sector that    │
│  recently showed interest in our solutions. Utilize all available data sources to compile a detailed profile,   │
│  focusing on key decision-makers, recent business developments, and potential needs that align with our         │
│  offerings. This task is crucial for tailoring our engagement strategy effectively.                             │
│  Don't make assumptions and only use information you absolutely sure about.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Sales Representative                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Enhancing AI Education at DeepLearningAI - A Partnership Proposal                                     │
│                                                                                                                 │
│  Dear Andrew Ng,                                                                                                │
│                                                                                                                 │
│  I hope this message finds you well. I am reaching out to you from CrewAI with great admiration for the recent  │
│  achievements of DeepLearningAI under your visionary leadership.                                                │
│                                                                                                                 │
│  I recently came across the news highlighting DeepLearningAI's groundbreaking efforts in AI education and       │
│  advancements, setting new standards in the industry. Your commitment to providing accessible courses and       │
│  driving global efforts in artificial intelligence education is truly inspiring.                                │
│                                                                                                                 │
│  At CrewAI, we specialize in providing innovative solutions that align perfectly with organizations like        │
│  DeepLearningAI, aiming to shape the future of artificial intelligence. Our advanced tools and strategies can   │
│  complement your initiatives, enhance learning experiences, and further propel DeepLearningAI to greater        │
│  heights.                                                                                                       │
│                                                                                                                 │
│  I believe that a partnership between DeepLearningAI and CrewAI could lead to remarkable collaborations,        │
│  driving innovation and excellence in the field of AI education. I would love the opportunity to discuss how    │
│  our tailored solutions can support your goals and contribute to the continued success of DeepLearningAI.       │
│                                                                                                                 │
│  Looking forward to the possibility of collaborating with you and the remarkable team at DeepLearningAI. Thank  │
│  you for considering this partnership proposal.                                                                 │
│                                                                                                                 │
│  Warm regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Lead Sales Representative                                                                                      │
│  CrewAI                                                                                                         │
│                                                                                                                 │
│  P.S. I have attached more information about CrewAI and

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 15953477-cbfc-4ead-b1c8-6a13583c6001                                                                     │
│  Agent: Lead Sales Representative                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bc358cb2-c8a6-4984-b76b-42d76398a0c7                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Subject: Enhancing AI Education at DeepLearningAI - A Partnership Proposal                       │
│                                                                                                                 │
│  Dear Andrew Ng,                                                                                                │
│                                                                                                                 │
│  I hope this message finds you well. I am reaching out to you from CrewAI with great admiration for the recent  │
│  achievements of DeepLearningAI under your visionary leadership.                                                │
│                                                                                                                 │
│  I recently came across the news highlighting DeepLearningAI's groundbreaking efforts in AI education and       │
│  advancements, setting new standards in the industry. Your commitment to providing accessible courses and       │
│  driving global efforts in artificial intelligence education is truly inspiring.                                │
│                                                                                                                 │
│  At CrewAI, we specialize in providing innovative solutions that align perfectly with organizations like        │
│  DeepLearningAI, aiming to shape the future of artificial intelligence. Our advanced tools and strategies can   │
│  complement your initiatives, enhance learning experiences, and further propel DeepLearningAI to greater        │
│  heights.                                                                                                       │
│                                                                                                                 │
│  I believe that a partnership between DeepLearningAI and CrewAI could lead to remarkable collaborations,        │
│  driving innovation and excellence in the field of AI education. I would love the opportunity to discuss how    │
│  our tailored solutions can support your goals and contribute to the continued success of DeepLearningAI.       │
│                                                                                                                 │
│  Looking forward to the possibility of collaborating with you and the remarkable team at DeepLearningAI. Thank  │
│  you for considering this partnership proposal.                                                                 │
│                                                                                                                 │
│  Warm regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Lead Sales Representative                                                                                      │
│  CrewAI                                                                                                         │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [15]:
from IPython.display import Markdown
Markdown(result.raw)

Subject: Enhancing AI Education at DeepLearningAI - A Partnership Proposal

Dear Andrew Ng,

I hope this message finds you well. I am reaching out to you from CrewAI with great admiration for the recent achievements of DeepLearningAI under your visionary leadership.

I recently came across the news highlighting DeepLearningAI's groundbreaking efforts in AI education and advancements, setting new standards in the industry. Your commitment to providing accessible courses and driving global efforts in artificial intelligence education is truly inspiring.

At CrewAI, we specialize in providing innovative solutions that align perfectly with organizations like DeepLearningAI, aiming to shape the future of artificial intelligence. Our advanced tools and strategies can complement your initiatives, enhance learning experiences, and further propel DeepLearningAI to greater heights.

I believe that a partnership between DeepLearningAI and CrewAI could lead to remarkable collaborations, driving innovation and excellence in the field of AI education. I would love the opportunity to discuss how our tailored solutions can support your goals and contribute to the continued success of DeepLearningAI.

Looking forward to the possibility of collaborating with you and the remarkable team at DeepLearningAI. Thank you for considering this partnership proposal.

Warm regards,

[Your Name]  
Lead Sales Representative  
CrewAI  

P.S. I have attached more information about CrewAI and how we can empower DeepLearningAI's mission. Feel free to reach out for any further details or to schedule a discussion at your convenience.